# 얼굴 검출(Detection) YOLO 파인튜닝 — 1단계 파이프라인

**범위:** 이 노트북은 "얼굴이 어디 있는지"만 검출하는 YOLO 모델을 학습한다 (단일 클래스 `face`).
"누구 얼굴인지" 식별하는 임베딩/등록(enrollment)/매칭 로직은 별도 모델이 담당하며 **이 노트북의
범위가 아니다** — 여기서 건드리지 않는다.

**하드웨어 제약:** WeGo LIMO 로봇 카메라가 사람 무릎보다 낮은 높이에 고정되어 있고 하드웨어 변경이
불가능하다. 즉 학습/검증 데이터 자체가 이미 이 낮은 앵글에서 찍힌 것이고, 이 노트북이 할 수 있는 건
그 조건 안에서 최대한 일반화되게 학습하는 것뿐이다 — 카메라 각도 문제 자체를 데이터 증강으로
"해결"할 수는 없다.

**데이터 특성:** person_A/person_B 각 약 132장, wp1/wp2/wp3(웨이포인트) x 조명on/off x
창문쪽/반대쪽 조합 x 조합당 11장 연속 burst로 촬영. 사실상 한 세션 안에서 찍혀 촬영 조건(조명/각도)
다양성이 적고, 추가 촬영도 불가능하다는 전제로 아래 파이프라인을 구성했다:
1. 데이터셋 자동 탐색 + 검증 (이미지/라벨 개수, 클래스 id, **burst 단위 train/val 분리 여부**)
2. burst가 train/val에 걸쳐 섞여 있으면 조합 단위로 재분할하는 스크립트
3. dataset.yaml 자동 생성 (`nc=1`, `names=['face']`)
4. 촬영 다양성 부족을 보완하는 데이터 증강 추가
5. `model.train()` → `best.pt`/`last.pt` → `best_model.val()` 재검증
6. 검증 결과 해석 시 소규모 데이터셋 주의 문구 출력

베이스 구조 참고: [WeGo LIMO 객체 인식 알고리즘 분석 (YOLO11n, 교통표지판/신호등)](https://humanoidsystem.kr/wego-limo-객체-인식-알고리즘-분석-yolo11n으로-교통-표지판과/) —
환경변수 분리, 데이터셋 자동 탐색, 개수 불일치 경고, YAML 자동 생성, train→검증 파이프라인 패턴을
그대로 가져오되, 클래스 구성/데이터 검증/증강/소규모 데이터 경고는 이 프로젝트에 맞게 다시 짰다.


## 1. 환경변수 설정

베이스 구조와 동일하게 `WORKSPACE_DIR`, `EPOCHS`, `IMGSZ`, `BATCH`, `WORKERS`, `DEVICE`를 환경변수로 분리한다. 데이터셋 경로는 이 프로젝트 기존 코드(`config.py`의 `CAREGUARD_FACE_DATASET_DIR`)와 동일한 관례를 따른다 — 새 관례를 만들지 않는다.

In [ ]:
import os
import sys
import csv
import glob
import shutil
import random
from pathlib import Path
from collections import defaultdict, Counter

import torch
import yaml

# 데이터셋 경로: 기존 careguard_perception/config.py의 FACE_DATASET_DIR와 같은 환경변수 관례.
# 실제 경로는 실행 환경에서 env var로 넘긴다 (예: 로봇에서 동기화해온 로컬 학습용 사본 경로).
WORKSPACE_DIR = Path(os.getenv(
    "WORKSPACE_DIR",
    os.getenv("CAREGUARD_FACE_DATASET_DIR", "~/careguard_data/face_dataset"),
)).expanduser().resolve()

RUNS_DIR = Path(os.getenv("RUNS_DIR", str(Path("~/careguard_data/runs").expanduser()))).resolve()

# YOLO_MODEL: 베이스 참고글은 yolo11n.pt 기본값을 씀. 이 저장소의 기존
# train_face_yolo.py 스모크테스트 스크립트는 yolov8n.pt를 기본값으로 쓰고 있으니,
# 기존 학습 결과와 비교하려면 YOLO_MODEL=yolov8n.pt 로 덮어써도 된다.
MODEL_NAME = os.getenv("YOLO_MODEL", "yolo11n.pt")
EPOCHS = int(os.getenv("EPOCHS", "100"))
IMGSZ = int(os.getenv("IMGSZ", "640"))
BATCH = int(os.getenv("BATCH", "8"))
WORKERS = int(os.getenv("WORKERS", "4"))
DEVICE = os.getenv("DEVICE", "0" if torch.cuda.is_available() else "cpu")
RUN_NAME = os.getenv("RUN_NAME", "face_detect_v1")

# burst(웨이포인트+조명+방향 조합) 매핑 CSV. person,idx,split,combo,filename 형식.
# 파일명만으로는 어떤 촬영 조합에 속하는지 알 수 없어서 별도 매핑이 필요하다
# (촬영 로그 기반으로 이미 만들어 둔 combo_corrected.csv가 있다면 그걸 가리키면 된다).
COMBO_CSV = Path(os.getenv("COMBO_CSV", str(WORKSPACE_DIR / "combo_corrected.csv")))

print(f"WORKSPACE_DIR = {WORKSPACE_DIR}")
print(f"RUNS_DIR      = {RUNS_DIR}")
print(f"MODEL_NAME    = {MODEL_NAME}")
print(f"EPOCHS={EPOCHS}  IMGSZ={IMGSZ}  BATCH={BATCH}  WORKERS={WORKERS}  DEVICE={DEVICE}")
print(f"COMBO_CSV     = {COMBO_CSV} (존재: {COMBO_CSV.is_file()})")


## 2. 데이터셋 경로 자동 탐색

`images/train`, `images/val`, `labels/train`, `labels/val` 네 폴더가 모두 있는 위치를 찾는다. `WORKSPACE_DIR` 바로 아래에 없으면 하위 디렉터리를 재귀 탐색한다. (베이스 구조에는 있던 dataset.zip 압축 해제 단계는 뺐다 — 이 프로젝트는 로봇/개발 PC에 이미 풀린 상태로 동기화해 쓰는 구조라 zip 아카이브 배포 방식이 아니다.)

In [ ]:
def is_yolo_dataset_root(root: Path) -> bool:
    return all((root / p).is_dir() for p in
               ("images/train", "images/val", "labels/train", "labels/val"))


def find_dataset_root(workspace: Path):
    candidates = [workspace, workspace / "face_dataset", workspace / "dataset"]
    for c in candidates:
        if is_yolo_dataset_root(c):
            return c
    if workspace.is_dir():
        for images_train in workspace.rglob("images/train"):
            root = images_train.parent.parent
            if is_yolo_dataset_root(root):
                return root
    return None


DATASET_ROOT = find_dataset_root(WORKSPACE_DIR)
if DATASET_ROOT is None:
    raise SystemExit(
        f"images/train, images/val, labels/train, labels/val 네 폴더를 모두 갖춘 데이터셋을 "
        f"{WORKSPACE_DIR} 아래에서 찾지 못했습니다. WORKSPACE_DIR(또는 CAREGUARD_FACE_DATASET_DIR) "
        f"환경변수를 확인하세요."
    )
print(f"데이터셋 루트: {DATASET_ROOT}")


## 3. 데이터 검증 — 이미지/라벨 개수 및 대응 여부

베이스 구조는 train/val 각각 이미지 개수 vs 라벨 개수만 비교했다. 여기서는 한 단계 더 나가서 **파일명(stem) 단위로 실제 짝이 맞는지**도 확인한다 — 이 프로젝트 데이터는 정리 과정에서 이미지/라벨 쌍이 어긋난 적이 있어서 개수만 맞아도 안심할 수 없다.

In [ ]:
IMG_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")


def list_stems(d: Path, exts):
    return {p.stem: p for p in d.iterdir() if p.suffix.lower() in exts} if d.is_dir() else {}


for split in ("train", "val"):
    img_dir = DATASET_ROOT / "images" / split
    lbl_dir = DATASET_ROOT / "labels" / split
    imgs = list_stems(img_dir, IMG_EXTS)
    lbls = list_stems(lbl_dir, (".txt",))

    print(f"[{split}] 이미지 {len(imgs)}장 / 라벨 {len(lbls)}개")
    if len(imgs) != len(lbls):
        print(f"  ⚠️  {split} 이미지 수와 라벨 수가 다릅니다 ({len(imgs)} vs {len(lbls)}).")

    missing_labels = sorted(set(imgs) - set(lbls))
    missing_images = sorted(set(lbls) - set(imgs))
    if missing_labels:
        print(f"  ⚠️  라벨이 없는 이미지 {len(missing_labels)}개 (예: {missing_labels[:5]})")
    if missing_images:
        print(f"  ⚠️  이미지가 없는 라벨 {len(missing_images)}개 (예: {missing_images[:5]})")
    if not missing_labels and not missing_images and len(imgs) == len(lbls):
        print("  OK: 이미지-라벨 1:1 대응 확인됨")


## 4. 클래스 id 검증

이번 1단계 파이프라인은 **`class 0 = face` 단일 클래스**여야 한다 (신원 구분은 별도 임베딩 모델 몫). 그런데 이 프로젝트에는 identity별로 class id를 부여하는 방식(`person_A=0`, `person_B=1`…)의 다른 라벨링 파이프라인도 존재한다. 실수로 그쪽 라벨을 가리키고 있지 않은지 여기서 확인한다.

In [ ]:
def collect_label_class_ids(labels_dir: Path) -> Counter:
    ids = Counter()
    for txt in labels_dir.glob("*.txt"):
        for line in txt.read_text().splitlines():
            line = line.strip()
            if not line:
                continue
            ids[int(line.split()[0])] += 1
    return ids


class_id_warning = False
for split in ("train", "val"):
    ids = collect_label_class_ids(DATASET_ROOT / "labels" / split)
    print(f"[{split}] class id 분포 = {dict(sorted(ids.items()))}")
    if len(ids) > 1:
        class_id_warning = True

if class_id_warning:
    print(
        "\n⚠️  라벨에 class id가 2개 이상 섞여 있습니다. 이 노트북은 \"얼굴 위치 검출\"만 하는 "
        "1단계 파이프라인 전제(class 0=face 단일 클래스)인데, identity별로 class id를 부여하는 "
        "다른 파이프라인의 라벨을 그대로 가리키고 있는 것일 수 있습니다. 학습 전에 (a) 모든 라벨의 "
        "class id를 0으로 통일할지, (b) 애초에 잘못된 데이터셋 경로를 가리키고 있는 건 아닌지 "
        "확인하세요. 확인 없이 그대로 학습하면 사실상 identity 분류 라벨로 \"얼굴 검출\"을 "
        "학습하게 되어 이번 1단계 목적과 어긋납니다."
    )


## 5. burst(웨이포인트+조명+방향 조합) 단위 train/val 분리 검증

같은 조합(wp x 조명 x 방향)은 11장 연속 burst로 촬영되어 프레임끼리 배경/조명/각도가 거의 동일하다. 이 burst가 train과 val에 걸쳐 섞여 있으면 val이 train과 사실상 같은 장면을 보게 되어 검증 성능이 과대평가된다. 파일명만으로는 어떤 조합인지 알 수 없으므로 `COMBO_CSV`
(`person,idx,split,combo,filename` 형식)를 읽어서 확인한다 — 이 CSV가 없으면 이 검증은 건너뛴다 (불가능한 걸 억지로 추정하지 않는다).

In [ ]:
def load_combo_rows(csv_path: Path):
    if not csv_path.is_file():
        return None
    with open(csv_path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


combo_rows = load_combo_rows(COMBO_CSV)

if combo_rows is None:
    print(f"[SKIP] combo 매핑 CSV를 찾지 못했습니다: {COMBO_CSV}")
    print("       burst 단위 train/val 분리 여부를 검증하려면 person,idx,split,combo,filename")
    print("       형식의 매핑 CSV를 COMBO_CSV 환경변수로 지정하세요. (이번 실행은 이 검증을 건너뜁니다)")
    leaking_groups = {}
else:
    groups = defaultdict(lambda: defaultdict(int))  # groups[(person, combo)][split] = count
    for r in combo_rows:
        groups[(r["person"], r["combo"])][r["split"]] += 1

    leaking_groups = {k: v for k, v in groups.items() if len(v) > 1}

    print(f"총 {len(groups)}개 (인물, 조합) 그룹 중 {len(leaking_groups)}개가 train/val에 걸쳐 섞여 있습니다.\n")
    if leaking_groups:
        print("⚠️  아래 조합은 같은 burst가 train/val에 나뉘어 들어가 있습니다 (val 성능 과대평가 위험):")
        for (person, combo), split_counts in sorted(leaking_groups.items()):
            detail = ", ".join(f"{s}={c}" for s, c in sorted(split_counts.items()))
            print(f"  - {person} / {combo}: {detail}")
        print("\n다음 셀의 재분할 스크립트로 조합을 쪼개지 말고 통째로 재분할하세요.")
    else:
        print("✅ 모든 조합이 train 또는 val 한쪽에만 속해 있습니다 (burst 누수 없음).")


## 6. burst 단위 재분할 스크립트

위에서 누수가 발견됐을 때 쓴다. **조합을 쪼개지 않고 통째로** train 또는 val 한쪽에만 배정되도록, 인물별로 조합 목록을 섞어서 지정 비율(`TARGET_VAL_RATIO`)만큼 val로 보낸다 — 기존 split 배정은 참고하지 않고 처음부터 다시 나눈다(패치가 아니라 깨끗한 재분할).

**안전장치:** 기존 `images/train`, `images/val`, `labels/train`, `labels/val`은 절대 건드리지 않는다. 결과는 `images/train_resplit` 등 별도 디렉터리에 복사본으로만 쓴다. 검토 후 아래처럼 직접 바꿔치기하세요:
```bash
mv images/train images/train_backup && mv images/train_resplit images/train
mv images/val   images/val_backup   && mv images/val_resplit   images/val
mv labels/train labels/train_backup && mv labels/train_resplit labels/train
mv labels/val   labels/val_backup   && mv labels/val_resplit   labels/val
```
(combo CSV의 각 행은 `person,split,idx`로 구분되어 있어서, 같은 파일명 stem이 train/val에 둘 다 존재하는 경우—실제로 이 데이터셋에서 있었던 일—에도 올바른 원본 파일을 정확히 찾는다.)

In [ ]:
EXECUTE_RESPLIT = False   # 실제로 파일을 복사하려면 True로. 기본은 계획만 출력하는 dry-run.
TARGET_VAL_RATIO = 0.2    # 인물별로 이 비율에 가깝게 "조합" 단위로 val에 배정
RESPLIT_SEED = 42


def resplit_by_combo(dataset_root: Path, combo_rows, val_ratio: float, seed: int, execute: bool):
    if not combo_rows:
        print("combo 매핑이 없어 재분할을 수행할 수 없습니다 (이전 셀 참고).")
        return

    groups = defaultdict(list)  # (person, combo) -> [row, row, ...]  (원래 split은 무시)
    for r in combo_rows:
        groups[(r["person"], r["combo"])].append(r)

    rng = random.Random(seed)
    plan = {}  # (person, combo) -> "train" | "val"
    per_person_keys = defaultdict(list)
    for key in groups:
        per_person_keys[key[0]].append(key)

    for person, keys in per_person_keys.items():
        keys = sorted(keys)
        rng.shuffle(keys)
        n_val = max(1, round(len(keys) * val_ratio))
        val_keys = set(keys[:n_val])
        for key in keys:
            plan[key] = "val" if key in val_keys else "train"
        print(f"  {person}: 조합 {len(keys)}개 중 {len(val_keys)}개 -> val")

    n_train_imgs = sum(len(groups[k]) for k, s in plan.items() if s == "train")
    n_val_imgs = sum(len(groups[k]) for k, s in plan.items() if s == "val")
    print(f"\n재분할 계획: train {n_train_imgs}장 / val {n_val_imgs}장")

    if not execute:
        print("[dry-run] EXECUTE_RESPLIT=True 로 바꾸면 *_resplit 디렉터리로 실제 복사합니다.")
        return

    for split in ("train", "val"):
        (dataset_root / "images" / f"{split}_resplit").mkdir(parents=True, exist_ok=True)
        (dataset_root / "labels" / f"{split}_resplit").mkdir(parents=True, exist_ok=True)

    copied, missing = 0, []
    for key, rows in groups.items():
        target_split = plan[key]
        for r in rows:
            fname = r["filename"]
            stem = Path(fname).stem
            img_src = dataset_root / "images" / r["split"] / fname
            lbl_src = dataset_root / "labels" / r["split"] / f"{stem}.txt"
            if not img_src.is_file() or not lbl_src.is_file():
                missing.append((str(img_src), img_src.is_file(), lbl_src.is_file()))
                continue
            shutil.copy2(img_src, dataset_root / "images" / f"{target_split}_resplit" / fname)
            shutil.copy2(lbl_src, dataset_root / "labels" / f"{target_split}_resplit" / f"{stem}.txt")
            copied += 1

    print(f"\n{copied}쌍 복사 완료 -> images/{{train,val}}_resplit, labels/{{train,val}}_resplit")
    if missing:
        print(f"⚠️  {len(missing)}개 항목은 원본을 찾지 못해 건너뛰었습니다 (combo CSV와 실제 파일이 "
              f"어긋난 경우 — 먼저 CSV를 최신 상태로 맞추세요):")
        for path, has_img, has_lbl in missing[:10]:
            print(f"   {path} (image={has_img}, label={has_lbl})")


if leaking_groups:
    resplit_by_combo(DATASET_ROOT, combo_rows, TARGET_VAL_RATIO, RESPLIT_SEED, EXECUTE_RESPLIT)
else:
    print("누수가 없어 재분할이 필요 없습니다 (또는 combo CSV가 없어 검증을 건너뛴 상태입니다).")


## 6b. ultralytics 라벨 캐시 무효화

ultralytics는 `labels/train.cache`, `labels/val.cache`에 스캔 결과(라벨 배열 자체)를 캐싱하고, 그 유효성 검사가 라벨 파일의 **실제 내용 변경**까지는 못 잡아낼 수 있다 — 클래스 id만 바꿔서 파일 크기가 거의 그대로인 경우 캐시가 구버전 라벨을 계속 재사용하는 걸 실제로 겪었다 (예: 위 재분할 스크립트로 `*_resplit`를 만든 뒤 바꿔치기하거나, 클래스 id를 수정한 뒤 곧바로 재학습할 때). 학습 직전에 항상 지우고 새로 스캔하게 한다.

In [ ]:
removed = list((DATASET_ROOT / "labels").glob("*.cache"))
for cache in removed:
    cache.unlink()
    print(f"삭제: {cache}")
print(f"라벨 캐시 정리 완료 ({len(removed)}개 삭제).")


## 7. dataset.yaml 생성

베이스 구조와 동일하게 학습 직전에 YAML을 자동 생성한다. 클래스는 8개(표지판/신호등)가 아니라 **1개(`face`)**로 바꿨다 — 신원 구분은 이 노트북 범위 밖.

In [ ]:
CLASS_NAMES = ["face"]  # 단일 클래스. person_A/B 구분은 별도 임베딩 모델 몫 (이 노트북 범위 아님).


def build_dataset_yaml(dataset_root: Path, names: list) -> Path:
    yaml_path = dataset_root / "face_dataset.yaml"
    yaml_path.write_text(yaml.dump({
        "path": str(dataset_root),
        "train": "images/train",
        "val": "images/val",
        "test": "",
        "nc": len(names),
        "names": names,
    }, allow_unicode=True, sort_keys=False))
    return yaml_path


DATA_YAML = build_dataset_yaml(DATASET_ROOT, CLASS_NAMES)
print(f"dataset yaml -> {DATA_YAML}\n")
print(DATA_YAML.read_text())


## 8. 데이터 증강 — 촬영 다양성 부족 보완

베이스 구조에는 증강 커스터마이징이 없었다. 이 데이터는 한 세션 안에서 찍혀 조명/각도 다양성이 적으므로 다음을 추가한다: **밝기/명암/색온도 랜덤 변화, 약간의 회전·크롭·스케일 변화, 모션 블러.** 옷/헤어스타일 관련 증강(예: cutout, copy-paste로 다른 사람 옷 합성 등)은 목적에 안 맞으므로 넣지 않는다.

- **회전/이동(크롭 효과)/스케일**은 ultralytics가 이미 잘 검증된 방식으로 bbox까지 맞춰 처리하는 `degrees`/`translate`/`scale` 하이퍼파라미터로 준다 (아래 학습 셀).
- **밝기/명암/색온도/모션 블러**는 ultralytics 하이퍼파라미터에 직접 노출된 옵션이 없어서, ultralytics가 내부적으로 쓰는 `Albumentations` 파이프라인을 확장하는 방식으로 추가한다 (라이브러리 코드를 고치는 게 아니라, 생성자를 몽키패치해서 우리 증강을 덧붙이는 방식 — ultralytics 공식 확장 지점은 아니지만 커뮤니티에서 흔히 쓰는 안전한 패턴이다: 좌표를 옮기는 변환은 전혀 포함하지 않으므로 bbox 처리 로직을 따로 신경 쓸 필요가 없다).

In [ ]:
try:
    import albumentations as A
    import ultralytics.data.augment as ultra_augment

    _face_albumentations_patched = globals().get("_face_albumentations_patched", False)
    if not _face_albumentations_patched:
        _orig_albumentations_init = ultra_augment.Albumentations.__init__

        def _patched_albumentations_init(self, p=1.0, transforms=None, flip_idx=None):
            # 설치된 ultralytics 버전의 실제 시그니처(p, transforms, flip_idx)를 그대로 받아
            # 원본 초기화에 넘긴다 — 여기서 transforms=None이면 원본이 자기 기본 목록을 만든다.
            _orig_albumentations_init(self, p=p, transforms=transforms, flip_idx=flip_idx)
            if self.transform is None:
                return  # albumentations 미설치 등으로 기본 파이프라인도 못 만든 경우
            if self.contains_spatial:
                # 우리가 추가하는 4개는 전부 픽셀값만 바꾸는 non-spatial 변환인데, 혹시 원본
                # 기본 목록이 spatial 변환을 포함하는 버전으로 바뀌면(bbox_params/keypoint_params
                # 조합이 달라짐) 안전하게 건너뛴다 — 잘못된 가정으로 bbox를 깨뜨리느니 증강 없이
                # 원본 동작을 그대로 두는 쪽이 낫다.
                print("[WARN] 기본 Albumentations 파이프라인이 spatial 변환을 포함해 커스텀 "
                      "밝기/명암/색온도/모션블러 증강을 건너뜁니다 (bbox 처리 방식이 달라 안전하지 않음).")
                return
            extra = [
                # 밝기/명암 랜덤 변화
                A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
                A.RandomGamma(gamma_limit=(80, 120), p=0.3),
                # 색온도(warm/cool) 변화 근사 — R/B 채널을 반대 방향으로 흔든다
                A.RGBShift(r_shift_limit=15, g_shift_limit=10, b_shift_limit=15, p=0.3),
                # 모션 블러 (주행 중 촬영 흔들림 대비)
                A.MotionBlur(blur_limit=(3, 9), p=0.25),
            ]
            # non-spatial 목록이므로 원본과 동일하게 bbox_params 없이 Compose (bbox 좌표는 안 건드림).
            self.transform = A.Compose(list(self.transform.transforms) + extra)

        ultra_augment.Albumentations.__init__ = _patched_albumentations_init
        _face_albumentations_patched = True

    print("커스텀 증강(밝기/명암/색온도/모션블러) 패치 적용됨 — model.train() 호출 시 자동 반영됩니다.")
except ImportError:
    print(
        "[WARN] albumentations 패키지가 없어 밝기/명암/색온도/모션블러 증강을 추가하지 못했습니다. "
        "기존 GPU/Docker 환경에 `pip install albumentations`만 추가하면 됩니다 (환경을 새로 만들 "
        "필요는 없습니다). 회전/이동/스케일 증강은 이 패키지 없이도 아래 학습 셀에서 그대로 적용됩니다."
    )


## 9. 학습

회전/이동/스케일은 여기서 하이퍼파라미터로 직접 준다. HSV 증강(`hsv_s`, `hsv_v`)도 기본값보다 키워서 밝기/색 다양성을 한 번 더 보강한다 (위 Albumentations 패치와 별개 경로 — 둘 다 걸리므로 중복이 아니라 상호 보완).

In [ ]:
from ultralytics import YOLO

model = YOLO(MODEL_NAME)

train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    workers=WORKERS,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=True,
    pretrained=True,
    verbose=True,
    # 약간의 회전 / 크롭(이동) / 스케일 변화 — 촬영 각도·거리 다양성 보완
    degrees=10.0,
    translate=0.08,
    scale=0.20,
    shear=0.0,
    perspective=0.0,
    # 밝기/색 다양성 보강 (Albumentations 패치와 상호 보완)
    hsv_h=0.015,
    hsv_s=0.5,
    hsv_v=0.4,
    fliplr=0.5,
    flipud=0.0,
)


## 10. best.pt / last.pt 확인

In [ ]:
run_dir = RUNS_DIR / RUN_NAME
best_pt = run_dir / "weights" / "best.pt"
last_pt = run_dir / "weights" / "last.pt"

for label, p in (("best.pt", best_pt), ("last.pt", last_pt)):
    if p.is_file():
        size_mb = p.stat().st_size / (1024 * 1024)
        print(f"{label}: {p}  ({size_mb:.1f} MB)")
    else:
        print(f"{label}: 찾을 수 없음 ({p})")


## 11. best.pt로 재검증

In [ ]:
best_model = YOLO(str(best_pt))
metrics = best_model.val(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=f"{RUN_NAME}_val",
    exist_ok=True,
)


## 12. 결과 해석 — 작은 데이터셋 규모에 대한 주의

베이스 참고글은 검증 이미지 13장으로 나온 mAP 0.995를 "제한된 검증 환경에서 나온 값"이라고 못박았다. 이 데이터셋(인당 약 132장, 대부분 한 세션에서 촬영)도 실제 val 규모/클래스별 인스턴스 수를 그때그때 계산해서 같은 취지의 경고를 출력한다 — 수치를 하드코딩하지 않고 방금 검증에 실제로 쓰인 값을 읽는다.

In [ ]:
n_val_images = len(list_stems(DATASET_ROOT / "images" / "val", IMG_EXTS))
val_class_ids = collect_label_class_ids(DATASET_ROOT / "labels" / "val")
n_val_instances = sum(val_class_ids.values())

print(f"mAP50    = {metrics.box.map50:.3f}")
print(f"mAP50-95 = {metrics.box.map:.3f}")
print(f"precision(mean) = {metrics.box.mp:.3f}")
print(f"recall(mean)    = {metrics.box.mr:.3f}")

print(f"""
{'='*70}
⚠️  검증 결과 해석 시 주의 — 작은 데이터셋 규모

이번 검증셋: {n_val_images}장 / 얼굴 라벨 {n_val_instances}개 (단일 클래스 face).
데이터 대부분이 wp1/wp2/wp3 x 조명on/off x 창문쪽/반대쪽 조합으로, 사실상 한두 세션 안에서
촬영된 것이라 배경/조명/각도 다양성이 제한적입니다.

위 mAP/precision/recall 수치가 높게 나왔더라도, 이는 "이 촬영 조건들 안에서의" 성능일 뿐이며
- 새로운 조명 조건 (촬영 당시 없던 시간대/날씨)
- 새로운 배경/장소 (실제 순찰 경로의 wp1/wp2/wp3 외 지점)
- 로봇이 실제로 이동하며 생기는 모션 블러, 다양한 거리/각도
에 대한 일반화 성능까지 보장하지는 않습니다. 배포 전 실제 로봇으로 다양한 조건에서
직접 눈으로 확인하는 것을 권장합니다 (수치만으로 최종 판단하지 말 것).
{'='*70}
""")
